# **Modelado con PySpark**

En esta sección se implementa el modelo de clasificación con PySpark utilizando `pyspark.ml.classification.RandomForestClassifier`.

Siguiendo la guía del proyecto, se emplea validación cruzada mediante `CrossValidator` y una grilla de hiperparámetros construida con `ParamGridBuilder`. Además, se evalúa el modelo mediante métricas comparables con las utilizadas en scikit-learn, incluyendo ROC AUC, precisión, F1-score y matriz de confusión.

In [1]:
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

from pyspark import StorageLevel

## **Configuración optimizada de la sesión Spark**

Antes de entrenar el modelo, se configura explícitamente la sesión de Spark con parámetros de paralelismo y memoria, siguiendo la guía del proyecto.

In [2]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("LendingClub_Optimized") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.memory.fraction", 0.8) \
    .config("spark.memory.storageFraction", 0.3) \
    .getOrCreate()

## **Carga del dataset completo**

El dataset se carga directamente en una `SparkSession`, manteniendo los datos en formato distribuido y evitando cualquier conversión a pandas o transferencia masiva al driver.

In [3]:
data_path = r"C:\Users\Daniel Rangel\Documents\MachineLearning\Data\accepted_2007_to_2018Q4.csv.gz"

df_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(data_path)
)

df_spark = df_spark.withColumn(
    "default",
    when(col("loan_status") == "Charged Off", 1).otherwise(0)
)

print("Número de filas:", df_spark.count())
print("Número de columnas:", len(df_spark.columns))

Número de filas: 2260701
Número de columnas: 152


## **Selección de variables**

Se utilizan las mismas variables seleccionadas en el flujo de scikit-learn, con el fin de mantener consistencia entre ambos enfoques y permitir una comparación más justa.

In [4]:
num_vars = [
    "loan_amnt",
    "int_rate",
    "fico_range_high",
    "annual_inc",
    "dti",
    "revol_util",
    "open_acc",
    "total_acc"
]

cat_vars = [
    "emp_length",
    "purpose",
    "home_ownership",
    "addr_state"
]

selected_vars = num_vars + cat_vars + ["default"]

df_spark_model = df_spark.select(*selected_vars)

for c in num_vars:
    df_spark_model = df_spark_model.withColumn(c, col(c).cast("double"))

print("Número de filas:", df_spark_model.count())
print("Número de columnas:", len(df_spark_model.columns))

Número de filas: 2260701
Número de columnas: 13


## **Pipeline de preprocesamiento**

Las variables categóricas se transforman mediante `StringIndexer` y `OneHotEncoder`, mientras que las variables numéricas se imputan antes de ser ensambladas en un vector de características.

Dado que el modelo posterior es un algoritmo basado en árboles, no se aplica escalado, ya que este tipo de modelos no depende de la escala de las variables.

In [5]:
indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep"
    )
    for col_name in cat_vars
]

encoder = OneHotEncoder(
    inputCols=[f"{col_name}_indexed" for col_name in cat_vars],
    outputCols=[f"{col_name}_encoded" for col_name in cat_vars],
    handleInvalid="keep"
)

imputer = Imputer(
    inputCols=num_vars,
    outputCols=[f"{c}_imputed" for c in num_vars]
).setStrategy("median")

assembler_inputs = [f"{c}_imputed" for c in num_vars] + [f"{col_name}_encoded" for col_name in cat_vars]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

preprocess_pipeline = Pipeline(stages=indexers + [encoder, imputer, assembler])

In [6]:
preprocess_model = preprocess_pipeline.fit(df_spark_model)
df_spark_prepared = preprocess_model.transform(df_spark_model)

## **Almacenamiento en memoria del DataFrame transformado**

Siguiendo la guía del proyecto, el DataFrame resultante del preprocesamiento se almacena en memoria antes del entrenamiento del modelo.

In [7]:
df_spark_prepared = df_spark_prepared.cache()
df_spark_prepared.select("default").show(5)

+-------+
|default|
+-------+
|      0|
|      0|
|      0|
|      0|
|      0|
+-------+
only showing top 5 rows



## División en entrenamiento y prueba

El conjunto de datos preparado se divide en entrenamiento y prueba con una proporción aproximada de 80/20, manteniendo los datos en formato DataFrame de Spark.

In [8]:
train_spark, test_spark = df_spark_prepared.randomSplit([0.8, 0.2], seed=42)

print("Filas train:", train_spark.count())
print("Filas test:", test_spark.count())

Filas train: 1808391
Filas test: 452310


La partición del dataset en PySpark produjo **1,808,391** observaciones para entrenamiento y **452,310** para prueba, lo cual corresponde aproximadamente a una división 80/20.

Dado que la función `randomSplit` no garantiza estratificación exacta, será necesario verificar posteriormente la distribución de la variable objetivo en ambos subconjuntos. No obstante, la partición obtenida resulta consistente con la proporción esperada y deja preparado el flujo para la etapa de modelado con validación cruzada.

## **Definición del modelo y validación cruzada**

Se configuró un modelo de `RandomForestClassifier` en PySpark utilizando la variable `default` como etiqueta y el vector `features` como conjunto de predictores.

La búsqueda de hiperparámetros se definió mediante `ParamGridBuilder`, considerando las siguientes combinaciones:

- `numTrees`: 10, 50 y 100
- `maxDepth`: 5, 10 y 15

En total, esto genera **9 combinaciones** de hiperparámetros. La selección del mejor modelo se realizará mediante `CrossValidator` con **2 folds**, utilizando **ROC AUC** como métrica de evaluación a través de `BinaryClassificationEvaluator`.

El uso de validación cruzada permite comparar de forma sistemática distintas configuraciones del modelo dentro del entorno distribuido de Spark, sin recurrir a iteraciones manuales sobre los hiperparámetros.

In [9]:
rf = RandomForestClassifier(
    labelCol="default",
    featuresCol="features",
    seed=42
)

param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 50, 100])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build()
)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="default",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_auc,
    numFolds=2,
    parallelism=2
)

## **Entrenamiento del modelo Random Forest en PySpark**

A continuación se entrena el modelo

In [10]:
start_train_spark = time.time()

cv_model = crossval.fit(train_spark)

train_time_spark = time.time() - start_train_spark

print(f"Tiempo total de entrenamiento en PySpark: {train_time_spark:.2f} segundos")

Tiempo total de entrenamiento en PySpark: 4996.80 segundos


## **Resultado del entrenamiento con validación cruzada**

El entrenamiento del modelo de Random Forest en PySpark, incluyendo la validación cruzada sobre la grilla de hiperparámetros definida, tomó aproximadamente **4996.80 segundos**, equivalentes a cerca de **83.28 minutos**.

Este tiempo incluye la evaluación de las 9 combinaciones de hiperparámetros mediante `CrossValidator` con 2 folds. En comparación con el flujo de scikit-learn, este resultado permitirá analizar no solo el desempeño predictivo, sino también el costo computacional de cada enfoque sobre el dataset completo.

In [11]:
best_rf_model = cv_model.bestModel

print("Mejor número de árboles:", best_rf_model.getNumTrees)
print("Mejor profundidad máxima:", best_rf_model.getOrDefault("maxDepth"))

Mejor número de árboles: 100
Mejor profundidad máxima: 15


## **Mejor configuración encontrada**

La validación cruzada en PySpark seleccionó como mejor modelo una configuración con:

- `numTrees = 100`
- `maxDepth = 15`

Esto indica que, dentro de la grilla evaluada, el modelo con mayor número de árboles y mayor profundidad disponible fue el que alcanzó el mejor desempeño según la métrica ROC AUC utilizada en la validación cruzada.

In [12]:
start_pred_spark = time.time()

predictions_spark = best_rf_model.transform(test_spark)

pred_time_spark = time.time() - start_pred_spark

print(f"Tiempo de predicción en PySpark: {pred_time_spark:.2f} segundos")

Tiempo de predicción en PySpark: 0.04 segundos


## **Predicción sobre el conjunto de prueba**

Una vez seleccionado el mejor modelo mediante validación cruzada, se realizaron predicciones sobre el conjunto de prueba.

El tiempo de predicción fue de aproximadamente **0.04 segundos**, lo cual muestra que, una vez entrenado el modelo, la etapa de inferencia en PySpark resulta muy rápida sobre el conjunto preparado.

In [13]:
roc_auc_spark = evaluator_auc.evaluate(predictions_spark)

print("ROC AUC en PySpark:", roc_auc_spark)

ROC AUC en PySpark: 0.6867323860184992


## **Evaluación inicial con ROC AUC**

El modelo de Random Forest en PySpark alcanzó un **ROC AUC de 0.6867** sobre el conjunto de prueba.

Este valor indica que el modelo posee una capacidad de discriminación moderada entre préstamos en default y no default. Aunque no representa una separación perfecta entre clases, sí sugiere que el modelo logra capturar parte de la señal predictiva presente en los datos.

In [14]:
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="default",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="default",
    predictionCol="prediction",
    metricName="f1"
)

precision_spark = evaluator_precision.evaluate(predictions_spark)
f1_spark = evaluator_f1.evaluate(predictions_spark)

print("Precision (weighted) en PySpark:", precision_spark)
print("F1-score en PySpark:", f1_spark)

Precision (weighted) en PySpark: 0.7755179376120379
F1-score en PySpark: 0.824740529634038


## **Precisión y F1-score**

El modelo de Random Forest en PySpark obtuvo una **precisión ponderada de 0.7755** y un **F1-score de 0.8247** sobre el conjunto de prueba.

Estas métricas muestran un desempeño global razonable del modelo, aunque deben interpretarse teniendo en cuenta el desbalance de clases presente en la variable objetivo. En particular, al tratarse de métricas ponderadas, su valor está influido en mayor medida por el comportamiento sobre la clase mayoritaria.

Por esta razón, resulta necesario complementar esta evaluación con la matriz de confusión, con el fin de analizar más claramente cómo se distribuyen los aciertos y errores entre las clases `default = 0` y `default = 1`.

In [15]:
confusion_spark = (
    predictions_spark
    .groupBy("default", "prediction")
    .count()
    .orderBy("default", "prediction")
)

confusion_spark.show()

+-------+----------+------+
|default|prediction| count|
+-------+----------+------+
|      0|       0.0|398320|
|      1|       0.0| 53990|
+-------+----------+------+



## **Interpretación de la matriz de confusión**

La matriz de confusión muestra que el modelo clasificó todas las observaciones del conjunto de prueba como pertenecientes a la clase `0`.

En particular:

- Los **398,320** casos reales de la clase `0` fueron clasificados correctamente.
- Los **53,990** casos reales de la clase `1` fueron clasificados incorrectamente como `0`.
- No se registró ninguna predicción de la clase positiva.

Este comportamiento indica que, al igual que en el flujo de scikit-learn, el modelo de Random Forest en PySpark no logra identificar préstamos en default bajo la configuración actual.

Por tanto, aunque el modelo presenta una capacidad de discriminación moderada en términos de ROC AUC, dicha capacidad no se traduce en predicciones útiles para la clase minoritaria cuando se utiliza el umbral de decisión por defecto. Esto confirma que el desbalance de clases sigue siendo un problema central en el modelado.

## **Conclusión de la sección**

El modelo de Random Forest implementado en PySpark logró entrenarse correctamente sobre el dataset completo en formato distribuido, utilizando validación cruzada con `ParamGridBuilder` y `CrossValidator`.

El mejor modelo seleccionado correspondió a `numTrees = 100` y `maxDepth = 15`, con un tiempo total de entrenamiento de aproximadamente **4996.80 segundos** y un tiempo de predicción de **0.04 segundos**.

En términos de desempeño, el modelo alcanzó un **ROC AUC de 0.6867**, una **precisión ponderada de 0.7755** y un **F1-score de 0.8247**. Sin embargo, la matriz de confusión mostró que el modelo no predijo ningún caso de la clase positiva, clasificando todas las observaciones como `default = 0`.

En consecuencia, los resultados de PySpark confirman el mismo problema observado en scikit-learn: el desbalance de clases impide que el modelo identifique correctamente la clase minoritaria bajo la configuración actual. Esto sugiere que, para ambos enfoques, será necesario considerar estrategias adicionales como pesos de clase, ajuste de umbral o técnicas de balanceo.